
## 1. Environment Setup

In [ ]:
import os
import sys
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Resolve project root and add src/ to path for any shared utilities.
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

# Consistent plot styling across all notebooks in this project.
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print(f"Project root: {PROJECT_ROOT}")
print("Environment loaded.")

In [ ]:
def get_engine():
    """
    Create and return a SQLAlchemy engine connected to the platos_pizza database.

    quote_plus percent-encodes the password before embedding it in the
    connection string. This handles special characters such as @, $, and #
    that would otherwise be misinterpreted as URL syntax delimiters.
    """
    password = quote_plus(os.getenv("DB_PASSWORD"))
    connection_string = (
        f"mysql+pymysql://{os.getenv('DB_USER')}:{password}"
        f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT', '3306')}/{os.getenv('DB_NAME')}"
    )
    return create_engine(connection_string, echo=False)


engine = get_engine()

# Confirm the connection is live before proceeding.
with engine.connect() as conn:
    result = conn.execute(text("SELECT DATABASE();"))
    db_name = result.scalar()

print(f"Connected to database: {db_name}")


## 2. Warehouse Verification

### 2.1 Row count verification

In [ ]:
# Expected row counts from the load script output.
# These are the values we are testing against.
EXPECTED_COUNTS = {
    "dim_pizza_type": 32,
    "dim_pizza":      96,
    "dim_date":       358,
    "dim_time":       16382,
    "fact_orders":    48620,
}

results = []

with engine.connect() as conn:
    for table, expected in EXPECTED_COUNTS.items():
        actual = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        status = "PASS" if actual == expected else "FAIL"
        results.append({"table": table, "expected": expected, "actual": actual, "status": status})

counts_df = pd.DataFrame(results)
print(counts_df.to_string(index=False))

if counts_df["status"].eq("FAIL").any():
    print("\nWARNING: One or more row count checks failed. Investigate before proceeding.")
else:
    print("\nAll row count checks passed.")

### 2.2 NULL checks on critical columns


In [ ]:
null_checks = [
    ("fact_orders",    "order_details_id"),
    ("fact_orders",    "order_id"),
    ("fact_orders",    "date_id"),
    ("fact_orders",    "time_id"),
    ("fact_orders",    "pizza_id"),
    ("fact_orders",    "quantity"),
    ("fact_orders",    "unit_price"),
    ("fact_orders",    "total_price"),
    ("dim_pizza",      "pizza_id"),
    ("dim_pizza",      "price"),
    ("dim_pizza_type", "pizza_type_id"),
    ("dim_pizza_type", "category"),
]

null_results = []

with engine.connect() as conn:
    for table, column in null_checks:
        null_count = conn.execute(
            text(f"SELECT COUNT(*) FROM {table} WHERE {column} IS NULL")
        ).scalar()
        status = "PASS" if null_count == 0 else "FAIL"
        null_results.append({"table": table, "column": column, "null_count": null_count, "status": status})

null_df = pd.DataFrame(null_results)
print(null_df.to_string(index=False))

if null_df["status"].eq("FAIL").any():
    print("\nWARNING: NULL values found in critical columns. Investigate before proceeding.")
else:
    print("\nAll NULL checks passed.")

### 2.3 Referential integrity check

In [ ]:
integrity_checks = {
    "fact -> dim_pizza":      "SELECT COUNT(*) FROM fact_orders f LEFT JOIN dim_pizza d ON f.pizza_id = d.pizza_id WHERE d.pizza_id IS NULL",
    "fact -> dim_pizza_type": "SELECT COUNT(*) FROM fact_orders f LEFT JOIN dim_pizza_type d ON f.pizza_type_id = d.pizza_type_id WHERE d.pizza_type_id IS NULL",
    "fact -> dim_date":       "SELECT COUNT(*) FROM fact_orders f LEFT JOIN dim_date d ON f.date_id = d.date_id WHERE d.date_id IS NULL",
    "fact -> dim_time":       "SELECT COUNT(*) FROM fact_orders f LEFT JOIN dim_time d ON f.time_id = d.time_id WHERE d.time_id IS NULL",
}

integrity_results = []

with engine.connect() as conn:
    for check_name, query in integrity_checks.items():
        orphan_count = conn.execute(text(query)).scalar()
        status = "PASS" if orphan_count == 0 else "FAIL"
        integrity_results.append({"check": check_name, "orphaned_rows": orphan_count, "status": status})

integrity_df = pd.DataFrame(integrity_results)
print(integrity_df.to_string(index=False))

if integrity_df["status"].eq("FAIL").any():
    print("\nWARNING: Referential integrity failures found. Investigate before proceeding.")
else:
    print("\nAll referential integrity checks passed.")


## 3. Data Quality Audit

### 3.1 Date range and operating period

In [ ]:
query = """
    SELECT
        MIN(date_id)                          AS first_order_date,
        MAX(date_id)                          AS last_order_date,
        DATEDIFF(MAX(date_id), MIN(date_id))  AS date_span_days,
        COUNT(DISTINCT date_id)               AS trading_days,
        COUNT(DISTINCT order_id)              AS total_orders,
        COUNT(*)                              AS total_line_items
    FROM fact_orders;
"""

with engine.connect() as conn:
    date_summary = pd.read_sql(text(query), conn)

print(date_summary.to_string(index=False))

### 3.2 Missing trading days

In [ ]:
query = """
    SELECT
        d.date_id,
        d.day_name,
        COUNT(f.order_id) AS order_count
    FROM dim_date d
    LEFT JOIN fact_orders f ON d.date_id = f.date_id
    GROUP BY d.date_id, d.day_name
    HAVING order_count = 0
    ORDER BY d.date_id;
"""

with engine.connect() as conn:
    zero_days = pd.read_sql(text(query), conn)

if zero_days.empty:
    print("No trading days with zero orders found.")
else:
    print(f"{len(zero_days)} day(s) with zero orders:")
    print(zero_days.to_string(index=False))

### 3.3 Time dimension granularity


In [ ]:
query = """
    SELECT
        t.meal_period,
        COUNT(f.order_details_id)  AS line_items,
        COUNT(DISTINCT f.order_id) AS unique_orders
    FROM fact_orders f
    JOIN dim_time t ON f.time_id = t.time_id
    GROUP BY t.meal_period
    ORDER BY line_items DESC;
"""

with engine.connect() as conn:
    meal_period_df = pd.read_sql(text(query), conn)

print(meal_period_df.to_string(index=False))

In [ ]:
# Define the expected period order and filter out any NaN values
# that may arise from unmatched time joins before plotting.
period_order = ["Morning", "Lunch", "Afternoon", "Dinner", "Late Night"]

plot_df = meal_period_df.dropna(subset=["meal_period"]).copy()
plot_df["meal_period"] = pd.Categorical(
    plot_df["meal_period"], categories=period_order, ordered=True
)
plot_df = plot_df.sort_values("meal_period")

palette = sns.color_palette("muted")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    plot_df["meal_period"].astype(str),
    plot_df["unique_orders"],
    color=[palette[0]] * len(plot_df)
)
ax.set_title("Unique Orders by Meal Period", fontsize=13, pad=12)
ax.set_xlabel("Meal Period")
ax.set_ylabel("Unique Orders")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
sns.despine()
plt.tight_layout()
plt.show()

### 3.4 Order quantity distribution


In [ ]:
query = """
    SELECT
        MIN(quantity)                          AS min_qty,
        MAX(quantity)                          AS max_qty,
        ROUND(AVG(quantity), 2)                AS avg_qty,
        SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END) AS zero_or_negative,
        SUM(CASE WHEN quantity > 10 THEN 1 ELSE 0 END) AS qty_over_10
    FROM fact_orders;
"""

with engine.connect() as conn:
    qty_summary = pd.read_sql(text(query), conn)

print(qty_summary.to_string(index=False))

In [ ]:
query = """
    SELECT quantity, COUNT(*) AS frequency
    FROM fact_orders
    GROUP BY quantity
    ORDER BY quantity;
"""

with engine.connect() as conn:
    qty_dist = pd.read_sql(text(query), conn)

# Cast quantity to string so matplotlib treats it as categorical,
# not continuous. Quantity values are discrete integers — rendering
# them on a continuous axis misrepresents the data.
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(
    qty_dist["quantity"].astype(str),
    qty_dist["frequency"],
    color=palette[1]
)
ax.set_title("Order Line Item Quantity Distribution", fontsize=13, pad=12)
ax.set_xlabel("Quantity")
ax.set_ylabel("Frequency")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
sns.despine()
plt.tight_layout()
plt.show()

print(qty_dist.to_string(index=False))

### 3.5 Price consistency audit

In [ ]:
query = """
    SELECT
        pizza_id,
        COUNT(DISTINCT unit_price) AS distinct_prices,
        MIN(unit_price)            AS min_price,
        MAX(unit_price)            AS max_price
    FROM fact_orders
    GROUP BY pizza_id
    HAVING COUNT(DISTINCT unit_price) > 1
    ORDER BY distinct_prices DESC;
"""

with engine.connect() as conn:
    price_issues = pd.read_sql(text(query), conn)

if price_issues.empty:
    print("Price consistency check passed. Every pizza_id has exactly one price.")
else:
    print(f"{len(price_issues)} pizza(s) found with inconsistent pricing:")
    print(price_issues.to_string(index=False))

### 3.6 Pizza size values

In [ ]:
query = """
    SELECT
        p.size,
        COUNT(f.order_details_id)             AS line_items,
        ROUND(SUM(f.total_price), 2)          AS total_revenue,
        ROUND(AVG(f.unit_price), 2)           AS avg_price
    FROM fact_orders f
    JOIN dim_pizza p ON f.pizza_id = p.pizza_id
    GROUP BY p.size
    ORDER BY line_items DESC;
"""

with engine.connect() as conn:
    size_df = pd.read_sql(text(query), conn)

print(size_df.to_string(index=False))

### 3.7 Menu coverage

In [ ]:
query = """
    SELECT
        p.pizza_id,
        pt.name,
        p.size,
        COUNT(f.order_details_id) AS times_ordered
    FROM dim_pizza p
    JOIN dim_pizza_type pt ON p.pizza_type_id = pt.pizza_type_id
    LEFT JOIN fact_orders f ON p.pizza_id = f.pizza_id
    GROUP BY p.pizza_id, pt.name, p.size
    HAVING times_ordered = 0
    ORDER BY pt.name, p.size;
"""

with engine.connect() as conn:
    unordered = pd.read_sql(text(query), conn)

if unordered.empty:
    print("Menu coverage check passed. Every pizza in the dimension table appears in at least one order.")
else:
    print(f"{len(unordered)} pizza(s) in the menu with zero orders:")
    print(unordered.to_string(index=False))

### 3.8 Duplicate order line item check

In [ ]:
query = """
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT order_details_id) AS unique_ids
    FROM fact_orders;
"""

with engine.connect() as conn:
    dup_check = pd.read_sql(text(query), conn)

total = dup_check["total_rows"].iloc[0]
unique = dup_check["unique_ids"].iloc[0]

if total == unique:
    print(f"Duplicate check passed. All {total:,} order_details_id values are unique.")
else:
    print(f"WARNING: {total - unique:,} duplicate order_details_id values detected.")


## 4. Findings Summary

In [ ]:
query = """
    SELECT
        COUNT(DISTINCT order_id)     AS total_orders,
        COUNT(*)                     AS total_line_items,
        ROUND(SUM(total_price), 2)   AS total_revenue,
        ROUND(AVG(total_price), 2)   AS avg_line_item_value,
        MIN(date_id)                 AS period_start,
        MAX(date_id)                 AS period_end
    FROM fact_orders;
"""

with engine.connect() as conn:
    summary = pd.read_sql(text(query), conn)

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"  Period:              {summary['period_start'].iloc[0]} to {summary['period_end'].iloc[0]}")
print(f"  Total orders:        {summary['total_orders'].iloc[0]:,}")
print(f"  Total line items:    {summary['total_line_items'].iloc[0]:,}")
print(f"  Total revenue:       ${summary['total_revenue'].iloc[0]:,.2f}")
print(f"  Avg line item value: ${summary['avg_line_item_value'].iloc[0]:,.2f}")

print("\n" + "=" * 60)
print("AUDIT FINDINGS")
print("=" * 60)
print("""
Finding 01 — Schema correction: size column
  The original schema defined size as CHAR(2). The dataset
  contains XXL (3 characters), which caused a load failure.
  Corrected to VARCHAR(5) before data was loaded.
  Impact: none after correction. All 96 pizza variants loaded cleanly.

Finding 02 — Time granularity: second-level precision
  Order times are recorded to the second, producing 16,382 unique
  time values across operating hours of 09:52 to 23:05.
  This is a data characteristic, not an error. All time-based
  analysis aggregates at the hour and meal_period level.

Finding 03 — Morning period has negligible volume
  Only 21 line items across 9 orders were placed before 11:00
  across the entire year (< 0.05% of total orders).
  The restaurant does not operate as a breakfast business.
  Morning data is excluded from peak-period analysis.

Finding 04 — XL and XXL are marginal products
  XL: 544 line items (1.1% of volume), $14,076 revenue.
  XXL: 28 line items (0.06% of volume), $1,007 revenue (~$2.76/day).
  Both sizes are candidates for menu rationalisation analysis
  in Notebook 02.

Finding 05 — Five menu items have zero orders
  The Big Meat Pizza (L, M), The Five Cheese Pizza (M, S), and
  The Four Cheese Pizza (S) appear in the menu but were never ordered.
  These SKUs occupy menu space with zero revenue contribution.
  Financial impact will be quantified in the revenue leakage analysis.

Finding 06 — Quantity is almost entirely single units
  98.1% of line items have a quantity of 1. Only 927 line items (1.9%)
  have a quantity of 2 or more, with a maximum of 4.
  Bulk ordering is effectively absent. This is relevant context for
  the upsell opportunity analysis in Notebook 02.
""")